In [18]:
from dash import Dash, html, dcc, Input, Output, callback_context
import plotly.express as px
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import numpy as np
import dash_bootstrap_components as dbc
from json import load
import chompjs
import re

In [11]:
df = pd.read_feather("2026_FV_Lasse_data.feather").reset_index()
df.head()

,index,DR1270,DR1275,DR1277,DR1278,DR1279,DR1280,DR1281,DR1285,DR1287,...,tv2-fv26-danmark-20,tv2-fv26-danmark-21,tv2-fv26-danmark-22,tv2-fv26-danmark-23,tv2-fv26-danmark-24,parti,navn,job,alder,kreds
0,0,4,1,5,2,2,1,4,2,5,...,0.0,0.75,1.00,0.0,0.25,alternativet,Martin Kjærulf,Technical Sultions Manager,47,fyns storkreds
1,1,5,1,5,1,2,1,1,2,5,...,0.0,0.50,1.00,0.0,0.00,alternativet,Jacob Holm,It-supporter,42,fyns storkreds
2,2,5,4,5,4,4,2,1,5,4,...,0.0,1.00,0.75,0.0,0.00,alternativet,Nana L. Buttenschøn,Virksomhedskonsulent,0,københavns omegns storkreds
3,3,4,2,5,5,2,1,4,2,2,...,0.0,0.25,0.75,0.0,0.75,alternativet,Peter Samir,Selvstændig,53,københavns omegns storkreds
4,4,5,2,5,2,2,1,1,5,5,...,0.0,1.00,1.00,0.0,0.00,alternativet,Tim Wodskou,Projektleder Novo Nordisk,48,københavns omegns storkreds


In [12]:
dk_spg = list(df.columns[df.columns.str.startswith("DR") | df.columns.str.startswith("tv2")])

In [13]:
color_dict = pd.read_json("various.json").apply(lambda x: x.str.lower()).set_index('bogstav_leg')['farver'].to_dict()


In [14]:
X = df[dk_spg]
y = df['parti']

lda = LinearDiscriminantAnalysis(n_components=2).fit(X, y)

q = pd.concat([
    df,
    pd.DataFrame(lda.transform(df[dk_spg]), columns=["X", "y"]).set_index(df.index)],
    #pd.DataFrame(PCA(n_components=2).fit_transform(X), columns=["X", "y"]).set_index(df.index)],
    axis=1)

In [15]:
bogfarve = pd.read_json("various.json").reset_index()
bogfarve['bogstav_leg'] = bogfarve['bogstav_leg'].str.lower()
bogfarve = bogfarve.set_index('bogstav_leg')
q['bogstav'] = q.parti.map(bogfarve['index']).fillna('X')

In [16]:
dr_sprg = load(open("./TakeTheDR/sprgs.json"))
dr_sprg

{'DR1270': 'De boligejere, der tjener mest på prisstigninger, skal betale mere i skat',
 'DR1275': 'Danmark bruger for mange penge på at støtte Ukraine i krigen mod Rusland',
 'DR1277': 'Det skal være forbudt for landbruget at sprøjte på sårbare drikkevandsområder, selv om det kan gøre produktionen af fødevarer dyrere',
 'DR1278': 'Atomkraft på dansk jord skal være en del af energiforsyningen på længere sigt',
 'DR1279': 'Når der skal sættes solceller op, bør hensyn til lokalbefolkningen veje tungere end grøn omstilling',
 'DR1280': 'Staten skal skære i støtten til Danmarks Radio',
 'DR1281': 'Det vil være bedst, hvis der dannes en regering hen over midten',
 'DR1285': 'Skatten for de højeste indkomster skal sættes op',
 'DR1287': 'Momsen på fødevarer skal sænkes',
 'DR1288': 'Det er okay, at den økonomiske ulighed stiger, så længe alle bliver rigere',
 'DR1289': 'Der skal være større udligning mellem rige og fattige kommuner',
 'DR1290': 'Prisen på cigaretter bør sættes markant op',
 

In [21]:
# tv2 spg data findes på deres hjemmeside i nemme at huske sti webpack/repo/components-tv2/src/components/Blocks/Election/CandidateTest/"data/questions"
with open('./TV2/fv2026.ts', 'r', encoding='utf-8') as f:
    content = f.read()

tv2_spg = chompjs.parse_js_object(re.search(r'=\s*(\[.*?\]);', content, re.DOTALL).group(1))
tv2_spg = {x['id']:x['question'] for x in tv2_spg if "danmark" in x['id']}
tv2_spg

{'tv2-fv26-danmark-1': 'Danmark bør bruge markant flere penge på forsvaret, end vi gør i dag, selvom det kan betyde færre penge til andre områder',
 'tv2-fv26-danmark-2': 'Det er en fordel for Danmark at være en del af EU',
 'tv2-fv26-danmark-3': 'Aktiv dødshjælp bør være lovligt i Danmark',
 'tv2-fv26-danmark-4': 'Transporttiden til specialiseret lægehjælp er blevet for lang, fordi hjælpen er samlet på få sygehuse',
 'tv2-fv26-danmark-5': 'Der opsættes for mange vindmøller og solceller på dansk jord',
 'tv2-fv26-danmark-6': 'Der bør etableres atomkraft i Danmark',
 'tv2-fv26-danmark-7': 'Hvis Grønland vil løsrive sig mere fra Kongeriget Danmark, bør Danmark skrue ned for den økonomiske støtte',
 'tv2-fv26-danmark-8': 'Danmark skal ikke købe nye våbensystemer fra USA',
 'tv2-fv26-danmark-9': 'Det er ikke et problem, at den økonomiske ulighed stiger, så længe danskerne samlet set bliver rigere',
 'tv2-fv26-danmark-10': 'Store bededag bør genindføres som helligdag, også selvom det betyde

In [25]:
sprgs = pd.Series(dr_sprg|tv2_spg)
sprgs

DR1270                 De boligejere, der tjener mest på prisstigning...
DR1275                 Danmark bruger for mange penge på at støtte Uk...
DR1277                 Det skal være forbudt for landbruget at sprøjt...
DR1278                 Atomkraft på dansk jord skal være en del af en...
DR1279                 Når der skal sættes solceller op, bør hensyn t...
DR1280                    Staten skal skære i støtten til Danmarks Radio
DR1281                 Det vil være bedst, hvis der dannes en regerin...
DR1285                  Skatten for de højeste indkomster skal sættes op
DR1287                                   Momsen på fødevarer skal sænkes
DR1288                 Det er okay, at den økonomiske ulighed stiger,...
DR1289                 Der skal være større udligning mellem rige og ...
DR1290                        Prisen på cigaretter bør sættes markant op
DR1292                 Det er vigtigere at investere i tog og busser ...
DR1293                          Afgifter på benzin 

In [26]:
#sprgs = sprgs.loc[dk_spg]
svar_muligheder = ['helt uenig','uenig','neutral','enig','helt enig']

In [27]:
def confidence_ellipse(x, y, n_std=1.96, size=100):
    if x.size != y.size:
        raise ValueError("x and y must be the same size")

    cov = np.cov(x, y)
    pearson = cov[0, 1]/np.sqrt(cov[0, 0] * cov[1, 1])
    ell_radius_x = np.sqrt(1 + pearson)
    ell_radius_y = np.sqrt(1 - pearson)
    theta = np.linspace(0, 2 * np.pi, size)
    ellipse_coords = np.column_stack([ell_radius_x * np.cos(theta), ell_radius_y * np.sin(theta)])
    x_scale = np.sqrt(cov[0, 0]) * n_std
    x_mean = np.mean(x)
    y_scale = np.sqrt(cov[1, 1]) * n_std
    y_mean = np.mean(y)  
    translation_matrix = np.tile([x_mean, y_mean], (ellipse_coords.shape[0], 1))
    rotation_matrix = np.array([[np.cos(np.pi / 4), np.sin(np.pi / 4)], [-np.sin(np.pi / 4), np.cos(np.pi / 4)]])
    scale_matrix = np.array([[x_scale, 0], [0, y_scale]])
    ellipse_coords = ellipse_coords.dot(rotation_matrix).dot(scale_matrix) + translation_matrix
    
    path = f'M {ellipse_coords[0, 0]}, {ellipse_coords[0, 1]}'
    for k in range(1, len(ellipse_coords)):
        path += f'L{ellipse_coords[k, 0]}, {ellipse_coords[k, 1]}'
    path += ' Z'
    return path

In [28]:
app = Dash(__name__, external_stylesheets=[dbc.themes.SOLAR, dbc.icons.BOOTSTRAP],
    meta_tags=[{"name": "viewport", "content": "width=device-width, initial-scale=1"},],
)

app.layout = dbc.Container([
	dcc.Store(id='bruger_coord'),
	dbc.Card(
		dbc.Container(
			[
				html.H2("Kommunalvalg 2025"),
				html.P("Analyse af hvor de enkelte kandidater står i forhold til hinanden og deres partier",
					   className="lead", ),
			],  # fluid=True,
		), body=True
	),
	dbc.Card(
		[
			dbc.Card([
                html.Label(["kreds filter:", dcc.Dropdown(id='kreds_valg', options=[{'value':'alle', 'label':'alle'}, *[{'value': x, 'label': x} for x in q.kreds.unique()]], value=['alle',], multi=True)]),
				dbc.Switch(id='parti_shadow', value=True, label="Tegn cirkler om partierne"),
				dbc.Switch(id='farveblind', value=False, label="Farveblind mode"),
			], body=True)
		],
	),

	dbc.Card(dcc.Graph(id='viz')),
	dbc.Card(html.P("(her kommer forudsigelser om hvilket parti en 'klikket' politiker burde være i)", id="svar_res")),
	dbc.Card([
		dcc.Markdown('''
		# SVAR
		### Tryk på politiker for at se deres svar eller svar selv for at se hvor DU ligger
		helt uenig  --  uenig  --  neutral  --  enig  --  helt enig
		'''),
		dbc.ListGroup([
			dbc.ListGroupItem([
				html.P(sprgs.loc[spg]),
				dcc.RadioItems(id=spg, options=[{'label': '', 'value': x / 4} for x in range(5)],
							   value=0, labelStyle={'display': 'inline-block'})
			]) for spg in dk_spg
		], flush=True)
	], body=True)
	, ],  # fluid=True
)

In [29]:
@app.callback(Output('viz', 'figure'), [Input('kreds_valg', 'value'),
	Input('parti_shadow', 'value'),
	Input('farveblind', 'value'),
	Input('bruger_coord', 'data')])
def update_graph(valgkreds_filter, shadow, farveblind, data):
    if 'alle' in valgkreds_filter:
        a = q
    else:
        a = q[q.kreds.isin(valgkreds_filter)]
        
    if farveblind:
        f1 = px.scatter(
            a, x='X', y='y', color='parti', color_discrete_map=color_dict, hover_data=['navn', 'job', 'alder'],
            custom_data=['index'], template="plotly_dark", labels={"X": "Hjalmesans", "y": "Fluplighed"},
            size='sized', text='bogstav', size_max=15
            # , width=1000  # , marginal_x='box'
        )
    else:
        f1 = px.scatter(
            a, x='X', y='y', color='parti', color_discrete_map=color_dict, hover_data=['navn', 'job', 'alder'],
            custom_data=['index'], template="plotly_dark", labels={"X": "Hjalmesans", "y": "Fluplighed"},
        )
        
    f1.layout.xaxis.fixedrange = True
    f1.layout.yaxis.fixedrange = True
    f1.update_layout(modebar_remove=['zoom', 'pan', 'select', 'lasso2d'])
    
    if shadow:
        for ii, (i, pari_data) in enumerate(q.groupby('parti')):
            if len(pari_data.X) > 100:
                f1.add_shape(type='path', path=confidence_ellipse(pari_data.X, pari_data.y), line_color='rgb(255,255,255,1)', fillcolor=color_dict[i], opacity=.4, )
    if data['dine_aktiv']:
        # f1.add_scatter(x=[data['dine_coords'][0]], y=[data['dine_coords'][1]], mode='markers', marker_symbol='star', marker_size=15)
        f1.add_vline(data['dine_coords'][0], line_dash="dash", line_color="pink")
        f1.add_hline(data['dine_coords'][1], line_dash="dash", line_color="pink")
    return f1

In [30]:
@app.callback([*[Output(x, 'value') for x in dk_spg], Output('svar_res', 'children'), Output('bruger_coord', 'data')],
			  {'clickData': Input('viz', 'clickData'), 'spg_in': [Input(x, 'value') for x in dk_spg]})
def display_click_data(clickData, spg_in):
	ctx = callback_context
	trigger_id = ctx.triggered[0]["prop_id"].split(".")[0]
	if trigger_id == 'viz':
		dine_aktiv = False
		if clickData and len(clickData['points']) != 0:
			idx = clickData['points'][0]['customdata'][0]
			navn = clickData['points'][0]['customdata'][1]
		else:
			idx = 1350
			navn = "(klik på nogen)"
		row = q[q['index'] == idx]
		parti = row['parti'].iloc[0]
		nyt_parti = lda.predict(row[dk_spg])[0]
		return [*[row[x].iloc[0] for x in dk_spg],
				f"Du har klikket på {navn}, {parti}. Vedkomne burde overveje {nyt_parti}",
				{'dine_aktiv': dine_aktiv, 'dine_coords': [0, 0]}]
	else:
		dine_aktiv = True
		a = pd.DataFrame(spg_in, index=dk_spg).T
		dine_coords = lda.transform(a)[0]
		return [*[x for x in spg_in],
				f"Dine koordinater er {dine_coords[0]:.1f}, {dine_coords[1]:.1f}. Du burde overveje {lda.predict(a)[0]}",
				{'dine_aktiv': dine_aktiv, 'dine_coords': dine_coords}]


In [31]:
#app.run_server(mode='jupyterlab')
#app.run_server(mode='external')
app.run(jupyter_mode="inline", jupyter_height=500, jupyter_width="100%")